# Reports — Part 4: Converting HL7 v2 `ORU^R01` into a FHIR Message

This is the fourth notebook in the series started by `01-fhir-search-basics.ipynb`.
`03-order-message-from-csv.ipynb` went from FHIR **to** HL7 v2 for an *order*. This
notebook goes the other way, for a *report*: HL7 v2 `ORU^R01` **into** FHIR, following
the [NW-GMSA HL7 v2 guidance](https://nw-gmsa.github.io/en/hl7v2.html) and, for the
underlying general segment/resource shape, HL7's own
[ORU_R01-to-Bundle ConceptMap](https://build.fhir.org/ig/HL7/v2-to-fhir/ConceptMap-message-oru-r01-to-bundle.html)
(NW-GMSA's own mapping is the one that actually governs this repo's data; the HL7
ConceptMap is a useful cross-check, not the authority here).

**The conversion step itself is hand-written Python, not a call to
`/transformToFHIR`** — that API is only used right at the end, to compare against what
this repo's real transformation engine produces for the same message. The point of doing
it by hand is that the segment-parsing is the *only* HL7-specific part. Once the data is
sitting in a plain dict, building the FHIR resources from it looks exactly the same
whether that dict came from parsing v2 segments or from a `SELECT` against a LIMS
database — which is the far more common real-world source for a report like this one.
Everywhere after the segment-parsing step, this notebook is really a "SQL rows → FHIR"
notebook that happens to source its rows from v2 text.

**Validation is slow** (each `validator_cli.jar` call reloads the whole IG, ~30-60s) —
fine here since we're doing it once per resource kind while developing, but not
something to do on every message in production. Treat the per-resource validation
cells below as a development-time habit, not a runtime step.

## The message

We're converting `Input/V2/R01/ctdna9737383222.txt` — a ctDNA report for Rob Leeds (NHS
number `9737383222`, MRN `RXR0817610` at Leeds Teaching Hospitals NHS Trust, ODS `RR8`).
Notice its `MSH-12` version is `2.3` and its identifiers are informal (a bare NHS number
in PID-2, no ODS-coded organisation anywhere) — this is what a lab's *outbound* feed
typically looks like before NW Genomics' Regional Integration Engine (RIE) standardises
it. Section 8 shows what the RIE actually sends Trusts instead: full MRNs, ODS-coded
organisations, NHS-number-typed identifiers — the same standardisation `03-order-message-from-csv.ipynb`
built by hand for orders.

![notebook 4 diagram 1](https://mermaid.ink/svg/Zmxvd2NoYXJ0IExSCiAgICBWMklOWyJITDcgdjIgT1JVXlIwMVxuKGZyb20gbGFiIC8gaUdlbmUpIl0KICAgIFNRTFsoIlNRTCBMSU1TIHRhYmxlc1xuKHJlYWwtd29ybGQgYWx0ZXJuYXRpdmUpIildCiAgICBST1dbInJlcG9ydF9yb3cgZGljdFxuKHNlY3Rpb25zIDItMykiXQogICAgRkhJUlsiRkhJUiBSMDEgQnVuZGxlXG5QYXRpZW50IC8gRW5jb3VudGVyIC8gU2VydmljZVJlcXVlc3Rcbk9ic2VydmF0aW9uIC8gQmluYXJ5IC8gRG9jdW1lbnRSZWZlcmVuY2UgLyBEaWFnbm9zdGljUmVwb3J0XG4oc2VjdGlvbnMgNC0xMCwgaGFuZC1idWlsdCArIHZhbGlkYXRlZCkiXQogICAgVjJPVVRbIlN0YW5kYXJkaXNlZCBITDcgdjIgT1JVXlIwMVxuZnVsbCBNUk4sIE9EUyBjb2RlcywgTkhTIG51bWJlclxuKHNlY3Rpb24gMTEpIl0KICAgIFRSVVNUWyJOSFMgVHJ1c3QgRVBSIl0KICAgIFQwMkZISVJbIkZISVIgVDAyIEJ1bmRsZVxuc2FtZSByZXNvdXJjZXMsIGV2ZW50Q29kaW5nIGZsaXBwZWRcbihzZWN0aW9uIDEzKSJdCiAgICBWMlQwMlsiSEw3IHYyIE1ETV5UMDJcbihzZWN0aW9uIDE0KSJdCiAgICBTQ1JWMlsiU2hhcmVkIGNhcmUgcmVjb3JkIHByb3ZpZGVyXG4oSEw3IHYyKSJdCiAgICBJVEkxMDVbIkZISVIgRG9jdW1lbnRSZWZlcmVuY2VcbklUSS0xMDUgU2ltcGxpZmllZCBQdWJsaXNoXG4oc2VjdGlvbiAxNSkiXQogICAgU0NSRkhJUlsiU2hhcmVkIGNhcmUgcmVjb3JkIHByb3ZpZGVyXG4oSUhFIE1IRCkiXQoKICAgIFYySU4gLS0+fCJoYW5kLXBhcnNlZCAowqcyLTMpInwgUk9XCiAgICBTUUwgLS4tPnwic2FtZSBzaGFwZSBlaXRoZXIgd2F5InwgUk9XCiAgICBST1cgLS0+IEZISVIKICAgIEZISVIgLS0+fHRyYW5zZm9ybVRvVjJ8IFYyT1VUIC0tPiBUUlVTVAogICAgRkhJUiAtLT58ImZsaXAgZXZlbnRDb2RpbmdcblIwMSAtPiBUMDIifCBUMDJGSElSCiAgICBUMDJGSElSIC0tPnx0cmFuc2Zvcm1Ub1YyfCBWMlQwMiAtLT4gU0NSVjIKICAgIFQwMkZISVIgLS0+fCJpbmxpbmUgQmluYXJ5LFxuYWRkIGhhc2gvc2l6ZSJ8IElUSTEwNSAtLT4gU0NSRkhJUg==)

<!--
```mermaid
flowchart LR
    V2IN["HL7 v2 ORU^R01\n(from lab / iGene)"]
    SQL[("SQL LIMS tables\n(real-world alternative)")]
    ROW["report_row dict\n(sections 2-3)"]
    FHIR["FHIR R01 Bundle\nPatient / Encounter / ServiceRequest\nObservation / Binary / DocumentReference / DiagnosticReport\n(sections 4-10, hand-built + validated)"]
    V2OUT["Standardised HL7 v2 ORU^R01\nfull MRN, ODS codes, NHS number\n(section 11)"]
    TRUST["NHS Trust EPR"]
    T02FHIR["FHIR T02 Bundle\nsame resources, eventCoding flipped\n(section 13)"]
    V2T02["HL7 v2 MDM^T02\n(section 14)"]
    SCRV2["Shared care record provider\n(HL7 v2)"]
    ITI105["FHIR DocumentReference\nITI-105 Simplified Publish\n(section 15)"]
    SCRFHIR["Shared care record provider\n(IHE MHD)"]

    V2IN -->|"hand-parsed (§2-3)"| ROW
    SQL -.->|"same shape either way"| ROW
    ROW --> FHIR
    FHIR -->|transformToV2| V2OUT --> TRUST
    FHIR -->|"flip eventCoding\nR01 -> T02"| T02FHIR
    T02FHIR -->|transformToV2| V2T02 --> SCRV2
    T02FHIR -->|"inline Binary,\nadd hash/size"| ITI105 --> SCRFHIR
```
-->

In [1]:
import base64
import hashlib
import json
import os
import subprocess
import tempfile
from datetime import datetime, timezone
from uuid import uuid4

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

toolsServer = os.getenv("V2_TOOLS")  # /transformToFHIR, /transformToV2 - used for comparison only, in sections 6-8

NHS_NUMBER_SYSTEM = "https://fhir.nhs.uk/Id/nhs-number"
ODS_SYSTEM = "https://fhir.nhs.uk/Id/ods-organization-code"
V2_0203 = "http://terminology.hl7.org/CodeSystem/v2-0203"
IGENE_REPORT_ID_SYSTEM = "https://fhir.nwgenomics.nhs.uk/iGene/ReportIdentifier"
GENOMIC_TEST_DIRECTORY_SYSTEM = "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory"
IGEAP_SYSTEM = "https://fhir.nwgenomics.nhs.uk/CodeSystem/IGEAP"
GENOMIC_CLINICAL_INDICATION_SYSTEM = "https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicClinicalIndication"
GENOMIC_TEST_OUTCOME_SYSTEM = "https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicTestOutcomeCode"
GLH_ODS = "699X0"
GLH_NAME = "NHS North West Genomics"

PATIENT_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Patient"
ENCOUNTER_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Encounter"
SERVICE_REQUEST_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/ServiceRequest"
GENOMIC_STUDY_PANEL_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/GenomicStudyPanel"
DIAGNOSTIC_REPORT_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/DiagnosticReport"
DOCUMENT_REFERENCE_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/DocumentReference"


def details(item):
    return item["text"]


def issues_df(outcome):
    df = pd.DataFrame(outcome["issue"])
    df["details"] = df["details"].apply(details)
    df.drop(columns=["extension"], inplace=True, errors="ignore")
    df.sort_values(by=["severity"], inplace=True)
    df = df[~df["details"].str.contains("ValueSet/mimetypes")]
    df = df[~df["details"].str.contains("failed: dom-6")]
    df = df[~df["details"].str.contains("bcp:13")]
    return df


def validate_resource(resource, profile):
    # Validate a single (non-bundled) resource against one NW-GMSA profile - a dev-loop
    # check, same pattern as 03-order-message-from-csv.ipynb.
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
        json.dump(resource, tmp)
        tmp_path = tmp.name
    outcome_path = tmp_path + "-OperationOutcome.json"

    subprocess.run(
        [
            "java", "-jar", "validator_cli.jar", tmp_path,
            "-version", "4.0.1", "-ig", "package.tgz",
            "-profile", profile, "-tx", "n/a",
            "-output", outcome_path, "-output-style", "json",
        ],
        capture_output=True,
    )

    with open(outcome_path) as f:
        outcome = json.load(f)
    return issues_df(outcome)

## 1. The source message

Every HL7 v2 test file in this repo uses CR line endings (`CLAUDE.md`), so splitting on
`\r` after normalising any `\r\n` gives us the raw segments.

In [2]:
with open("Input/V2/R01/ctdna9737383222.txt", newline="") as f:
    raw_v2 = f.read()

segments = [s for s in raw_v2.replace("\r\n", "\r").split("\r") if s]
for segment in segments:
    print(segment[:120])

MSH|^~\&|IGENE|MFT|EPIC|MFT|20260709130251||ORU^R01|c61c188b-4843-4b1f-8c84-12af80be0568|T|2.3
PID||9737383222|RXR0817610^^^RR8^MR||LEEDS^Rob||19780117|M|||^^^^LS1 3EX
PV1||U|||||||||||||||||SP26-01847
ORC|RE|1234-RR8|||||||||||||||||||Leeds Teaching Hospitals NHS Trust^^RR8^^^ODS
OBR|1|1234-RR8|T26-59X2|ctDNA_M4^PACKAGE: M4.14 - Non-Small Cell Lung Cancer, Multi-target ctDNA combined Multi-target N
OBX|1|CE|ctDNA_M4^PACKAGE: M4.14 - Non-Small Cell Lung Cancer, Multi-target ctDNA combined Multi-target NGS panel - smal
OBX|2|CE|51968-6^^LN|1|971^FAILURE|||||||||20260729103726+0000
NTE|1|L|M4.14=PACKAGE: M4.14 - Non-Small Cell Lung Cancer, Multi-target ctDNA combined Multi-target NGS panel - small va


## 2. Segments → fields

The only genuinely HL7-specific code in this whole notebook is this cell: split each
segment on `|` and index into it by segment type. Everything from section 3 onward
works off plain Python values and doesn't know or care that they came from pipe-delimited
text.

In [3]:
by_segment = {}
for segment in segments:
    fields = segment.split("|")
    by_segment.setdefault(fields[0], []).append(fields)

msh = by_segment["MSH"][0]
pid = by_segment["PID"][0]
pv1 = by_segment["PV1"][0]
orc = by_segment["ORC"][0]
obr = by_segment["OBR"][0]
obx_list = by_segment["OBX"]
nte = by_segment["NTE"][0]

## 3. Fields → a row

This is the point where a real SQL-backed system would already be starting from a dict
like the one below — one row from a query joining an orders table, a patient table, and
a results table. Building it here from v2 fields instead just means each value's source
comment says `OBR-4` instead of a column name.

A couple of fields need a second look:

- **`report_directory_code`** doesn't come from `OBR` at all — `NTE-3` here is
  `"M4.14=PACKAGE: ..."`, a `<Genomic Test Directory code>=<description>` pair. The
  clinical indication code used elsewhere (`ServiceRequest.reasonCode`,
  `Observation.component`) is the part of `M4.14` before the dot (`M4`) — the test
  *family*, not the specific test.
- **The embedded PDF** sits inside `OBX-1`'s `OBX-5`, itself abusing the `CE` (Coded
  Element) data type to carry `<blank>^<label>^<mime type>^Base64^<data>` rather than
  the more standard `ED` (Encapsulated Data) type — `OBX-4` (`"PDF"`) is what actually
  flags this OBX as the one carrying the report attachment, not `OBX-2`.
- **`51968-6`/`971`/`FAILURE`** (`OBX-2`) is the genomic test's own pass/fail-style
  outcome code, unrelated to `OBR-25`'s v2 result status (`F` = *Final*, i.e. the report
  itself is complete) — the two "F"/"FAILURE"-shaped values answer different questions
  and both matter.

In [4]:
def component(field, index, default=""):
    # .strip(): OBR-4's text component is truncated mid-word with a trailing space in
    # this fixture - v2 fields routinely carry incidental padding like this, and it's
    # not meaningful data, so trim it the way this repo's real transform engine does.
    parts = field.split("^")
    return parts[index].strip() if index < len(parts) else default


test_directory_code = nte[3].split("=")[0]

pdf_obx = next(o for o in obx_list if o[4] == "PDF")
outcome_obx = next(o for o in obx_list if o[3].startswith("51968-6"))

report_row = {
    # Patient (PID)
    "nhs_number": pid[2],
    "mrn": component(pid[3], 0),
    "mrn_assigner_ods": component(pid[3], 3),
    "family_name": component(pid[5], 0),
    "given_name": component(pid[5], 1),
    "birth_date": datetime.strptime(pid[7], "%Y%m%d").strftime("%Y-%m-%d"),
    "sex": pid[8],
    "postcode": component(pid[11], 4),
    # Encounter (PV1)
    "account_number": pv1[19],
    # Order (ORC/OBR)
    "placer_order_number": orc[2],
    "ordering_org_name": component(orc[21], 0),
    "ordering_org_ods": component(orc[21], 2),
    "filler_report_number": obr[3],
    "test_code_local": component(obr[4], 0),
    "test_description": component(obr[4], 1),
    "report_datetime": datetime.strptime(obr[22], "%Y%m%d%H%M%S").strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "result_status": obr[25],  # F = Final
    "interpreter_family": component(obr[32], 1),
    "interpreter_given": component(obr[32], 2),
    # Genomic Test Directory (NTE)
    "test_directory_code": test_directory_code,
    "clinical_indication_code": test_directory_code.split(".")[0],
    # Embedded PDF (OBX with OBX-4 == "PDF")
    "pdf_content_type": component(pdf_obx[5], 2),
    "pdf_base64": component(pdf_obx[5], 4),
    # Genomic test outcome (OBX with OBX-3 starting 51968-6, a LOINC code)
    "outcome_code": component(outcome_obx[5], 0),
    "outcome_display": component(outcome_obx[5], 1),
}

{k: (v[:40] + "..." if k == "pdf_base64" else v) for k, v in report_row.items()}

{'nhs_number': '9737383222',
 'mrn': 'RXR0817610',
 'mrn_assigner_ods': 'RR8',
 'family_name': 'LEEDS',
 'given_name': 'Rob',
 'birth_date': '1978-01-17',
 'sex': 'M',
 'postcode': 'LS1 3EX',
 'account_number': 'SP26-01847',
 'placer_order_number': '1234-RR8',
 'ordering_org_name': 'Leeds Teaching Hospitals NHS Trust',
 'ordering_org_ods': 'RR8',
 'filler_report_number': 'T26-59X2',
 'test_code_local': 'ctDNA_M4',
 'test_description': 'PACKAGE: M4.14 - Non-Small Cell Lung Cancer, Multi-target ctDNA combined Multi-target NGS panel - small variant (EGFR, ALK, BRAF, KRAS, MET exon 14 skipping and copy number variations) and structural variant (ROS1, RET, ALK, NTRK1, NTRK2, NTRK3, MET exon',
 'report_datetime': '2026-07-14T15:59:16+00:00',
 'result_status': 'F',
 'interpreter_family': 'Edgerley',
 'interpreter_given': 'Jonathan',
 'test_directory_code': 'M4.14',
 'clinical_indication_code': 'M4',
 'pdf_content_type': 'application/pdf',
 'pdf_base64': 'JVBERi0xLjQKMSAwIG9iago8PC9UeXBlIC9DYX

## 4. Patient

Same [`Patient` profile](https://nw-gmsa.github.io/en/StructureDefinition-Patient.html)
as `03-order-message-from-csv.ipynb`. This particular message's NHS number sits in
`PID-2` unlabelled rather than `PID-3` — just a v2-side quirk of this older-style
(`MSH-12` = `2.3`) feed; the FHIR `Patient` looks identical either way.

In [5]:
patient_id = str(uuid4())
patient_fullurl = f"urn:uuid:{patient_id}"

patient = {
    "resourceType": "Patient",
    "identifier": [
        {
            "system": NHS_NUMBER_SYSTEM,
            "type": {"coding": [{"system": V2_0203, "code": "NH"}]},
            "value": report_row["nhs_number"],
        },
        {
            "assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["mrn_assigner_ods"]}},
            "type": {"coding": [{"system": V2_0203, "code": "MR"}]},
            "value": report_row["mrn"],
        },
    ],
    "name": [{"family": report_row["family_name"], "given": [report_row["given_name"]]}],
    "gender": {"M": "male", "F": "female"}.get(report_row["sex"], "unknown"),
    "birthDate": report_row["birth_date"],
    "address": [{"postalCode": report_row["postcode"]}],
}

print(json.dumps(patient, indent=2))

{
  "resourceType": "Patient",
  "identifier": [
    {
      "system": "https://fhir.nhs.uk/Id/nhs-number",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "NH"
          }
        ]
      },
      "value": "9737383222"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "RR8"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "MR"
          }
        ]
      },
      "value": "RXR0817610"
    }
  ],
  "name": [
    {
      "family": "LEEDS",
      "given": [
        "Rob"
      ]
    }
  ],
  "gender": "male",
  "birthDate": "1978-01-17",
  "address": [
    {
      "postalCode": "LS1 3EX"
    }
  ]
}


In [6]:
validate_resource(patient, PATIENT_PROFILE)

,severity,code,details,expression


## 5. Encounter

The [`Encounter` profile](https://nw-gmsa.github.io/en/StructureDefinition-Encounter.html)
carries the account/visit number from `PV1-19` as an `AN`-typed identifier — this
message has almost nothing else in `PV1` to work with (no ward, no admit date), which
is normal for a report rather than an admission message.

In [7]:
encounter_id = str(uuid4())
encounter_fullurl = f"urn:uuid:{encounter_id}"

encounter = {
    "resourceType": "Encounter",
    "status": "finished",
    "class": {"system": "http://terminology.hl7.org/CodeSystem/v3-ActCode", "code": "OBSENC"},
    "identifier": [{"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]}],
    "subject": {
        "reference": patient_fullurl,
        "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]},
    },
}

print(json.dumps(encounter, indent=2))

{
  "resourceType": "Encounter",
  "status": "finished",
  "class": {
    "system": "http://terminology.hl7.org/CodeSystem/v3-ActCode",
    "code": "OBSENC"
  },
  "identifier": [
    {
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "AN"
          }
        ]
      },
      "value": "SP26-01847"
    }
  ],
  "subject": {
    "reference": "urn:uuid:6e347049-b169-4f19-b3b6-69908b0169dc",
    "identifier": {
      "system": "https://fhir.nhs.uk/Id/nhs-number",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "NH"
          }
        ]
      },
      "value": "9737383222"
    }
  }
}


In [8]:
validate_resource(encounter, ENCOUNTER_PROFILE)

,severity,code,details,expression
0,information,informational,This element does not match any known slice de...,[Encounter.identifier[0]]
1,information,business-rule,Reference to draft CodeSystem http://terminolo...,[Encounter.class]


The `information`/`warning` rows from here through the rest of this notebook's
individual-resource checks are the same `-tx n/a` terminology-server limitation
`03-order-message-from-csv.ipynb` covered — the validator flagging codes it can't check
without a terminology server, not defects. The two genuine `error`-severity rows further
down (sections 6 and 9) get their own explanation, since those are worth pausing on.

## 6. ServiceRequest — the order this report answers

`DiagnosticReport.basedOn` (built in section 9) will point back at this. Unlike
`03-order-message-from-csv.ipynb`'s order, this `ServiceRequest` is `completed` — by the
time a report exists, the order it came from is done. `code` carries only the Genomic
Test Directory coding (`M4.14`); the lab's own local `IGEAP` coding is added to
`DiagnosticReport.code` instead (section 9), not here.

In [9]:
service_request_id = str(uuid4())
service_request_fullurl = f"urn:uuid:{service_request_id}"

service_request = {
    "resourceType": "ServiceRequest",
    "status": "completed",
    "intent": "order",
    "category": [{"coding": [{"system": "http://snomed.info/sct", "code": "116148004"}]}],
    "code": {"coding": [{"system": GENOMIC_TEST_DIRECTORY_SYSTEM, "code": report_row["test_directory_code"]}]},
    "identifier": [
        {
            "assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}},
            "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]},
            "value": report_row["placer_order_number"],
        },
        {
            "assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}},
            "system": IGENE_REPORT_ID_SYSTEM,
            "type": {"coding": [{"system": V2_0203, "code": "FILL"}]},
            "value": report_row["filler_report_number"],
        },
    ],
    "subject": {
        "reference": patient_fullurl,
        "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]},
    },
    "requester": {
        "display": report_row["ordering_org_name"],
        "identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]},
        "type": "Organization",
    },
    "reasonCode": [{"coding": [{"system": GENOMIC_CLINICAL_INDICATION_SYSTEM, "code": report_row["clinical_indication_code"]}]}],
    "encounter": {
        "reference": encounter_fullurl,
        "identifier": {"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]},
    },
}

print(json.dumps(service_request, indent=2))

{
  "resourceType": "ServiceRequest",
  "status": "completed",
  "intent": "order",
  "category": [
    {
      "coding": [
        {
          "system": "http://snomed.info/sct",
          "code": "116148004"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
        "code": "M4.14"
      }
    ]
  },
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "RR8"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"
          }
        ]
      },
      "value": "1234-RR8"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699X0"
        }
      },
      "system": "https://fhir.nwgenomics.nhs.uk

In [10]:
validate_resource(service_request, SERVICE_REQUEST_PROFILE)

,severity,code,details,expression
1,error,structure,ServiceRequest.authoredOn: minimum required = ...,[ServiceRequest]
0,information,informational,This element does not match any known slice de...,[ServiceRequest.identifier[1]]
4,information,code-invalid,None of the codings provided are in the value ...,[ServiceRequest.code]
2,warning,code-invalid,Unknown Code 'M4.14' in the CodeSystem 'https:...,[ServiceRequest.code.coding[0].code]
3,warning,not-found,A definition for the value Set 'http://hl7.org...,[ServiceRequest.code]
5,warning,code-invalid,Unknown Code 'M4' in the CodeSystem 'https://f...,[ServiceRequest.reasonCode[0].coding[0].code]


The `error`-severity row (`ServiceRequest.authoredOn: minimum required = 1, but only
found 0`) looks alarming, but it isn't specific to our hand-built version: the *same*
`ServiceRequest`, pulled straight out of this repo's real `/transformToFHIR` output for
this exact message and validated the same way, produces the identical error. `authoredOn`
is `1..1` on this profile, but nothing in the source v2 (`ORC-9`, the usual home for it,
is blank here) gives either the real engine or us a value to put there. A genuine
pre-existing gap in the production pipeline for this message shape, not a mistake in
this notebook's conversion.

## 7. Observation — the genomic study panel result

The [`GenomicStudyPanel` profile](https://nw-gmsa.github.io/en/StructureDefinition-GenomicStudyPanel.html)
is a specialisation of `Observation` (`meta.profile`), not a separate resource type. Its
top-level `code` (`81306-3`, "Variables that apply to the overall study") is fixed for
this panel-level observation; the two per-message facts — which clinical indication
prompted the test, and how the test actually turned out — live in `component`, each with
its own LOINC code.

In [11]:
observation_id = str(uuid4())
observation_fullurl = f"urn:uuid:{observation_id}"

observation = {
    "resourceType": "Observation",
    "meta": {"profile": [GENOMIC_STUDY_PANEL_PROFILE]},
    "status": "final",
    "category": [
        {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "laboratory"}]},
        {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]},
    ],
    "code": {"coding": [{"system": "http://loinc.org", "code": "81306-3", "display": "Variables that apply to the overall study"}]},
    "identifier": [
        {
            "assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}},
            "system": IGENE_REPORT_ID_SYSTEM,
            "type": {"coding": [{"system": V2_0203, "code": "FILL"}]},
            "value": report_row["filler_report_number"],
        }
    ],
    "subject": {
        "reference": patient_fullurl,
        "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]},
    },
    "effectiveDateTime": report_row["report_datetime"],
    "component": [
        {
            "code": {"coding": [{"system": "http://loinc.org", "code": "51967-8"}]},
            "valueCodeableConcept": {"coding": [{"system": GENOMIC_CLINICAL_INDICATION_SYSTEM, "code": report_row["clinical_indication_code"]}]},
        },
        {
            "code": {"coding": [{"system": "http://loinc.org", "code": "51968-6"}]},
            "valueCodeableConcept": {
                "coding": [{"system": GENOMIC_TEST_OUTCOME_SYSTEM, "code": report_row["outcome_code"], "display": report_row["outcome_display"]}]
            },
        },
    ],
}

print(json.dumps(observation, indent=2))

{
  "resourceType": "Observation",
  "meta": {
    "profile": [
      "https://fhir.nwgenomics.nhs.uk/StructureDefinition/GenomicStudyPanel"
    ]
  },
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "laboratory"
        }
      ]
    },
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/v2-0074",
          "code": "GE"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "81306-3",
        "display": "Variables that apply to the overall study"
      }
    ]
  },
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699X0"
        }
      },
      "system": "https://fhir.nwgenomics.nhs.uk/iGene/ReportIdentifier",
      "type": {
        "coding": [
          

In [12]:
validate_resource(observation, GENOMIC_STUDY_PANEL_PROFILE)

,severity,code,details,expression
0,information,informational,This element does not match any known slice de...,[Observation.component[0]]
1,information,informational,This element does not match any known slice de...,[Observation.component[1]]
2,information,informational,This element does not match any known slice de...,[Observation.component[0].value.ofType(Codeabl...
3,information,business-rule,Reference to draft CodeSystem https://fhir.nwg...,[Observation.component[0].value.ofType(Codeabl...
5,information,informational,This element does not match any known slice de...,[Observation.component[1].value.ofType(Codeabl...
6,information,business-rule,Reference to draft CodeSystem https://fhir.nwg...,[Observation.component[1].value.ofType(Codeabl...
4,warning,code-invalid,Unknown Code 'M4' in the CodeSystem 'https://f...,[Observation.component[0].value.ofType(Codeabl...
8,warning,invalid,"Best Practice Recommendation: In general, all ...",[Observation]


## 8. Binary + DocumentReference — the embedded PDF

The PDF itself becomes a standalone `Binary` (base64 in `.data`), referenced from
`DocumentReference.content.attachment.url` by `urn:uuid` — never inlined directly into
`DocumentReference`. `DiagnosticReport.presentedForm` (section 9) points at the same
`Binary`, so the PDF exists exactly once in the bundle even though two resources refer
to it.

`DocumentReference.type` needs a SNOMED or LOINC code, but the only code this OBX
actually supplies (`ctDNA_M4`, `IGENE`) is neither — this repo's own transformation
engine substitutes SNOMED `1054161000000101` "Genetic report" in that situation (see
`IntegrationTest.py`'s `check_document_reference_code`, and the same code already used
for this same message's `T02` fixture), so we do the same rather than inventing a
different convention.

In [13]:
binary_id = str(uuid4())
binary_fullurl = f"urn:uuid:{binary_id}"

binary = {
    "resourceType": "Binary",
    "contentType": report_row["pdf_content_type"],
    "data": report_row["pdf_base64"],
}

document_reference_id = str(uuid4())
document_reference_fullurl = f"urn:uuid:{document_reference_id}"

document_reference = {
    "resourceType": "DocumentReference",
    "status": "current",
    "type": {"coding": [{"system": "http://snomed.info/sct", "code": "1054161000000101", "display": "Genetic report"}]},
    "subject": {
        "reference": patient_fullurl,
        "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]},
    },
    "date": report_row["report_datetime"],
    "identifier": [
        {
            "assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}},
            "system": IGENE_REPORT_ID_SYSTEM,
            "type": {"coding": [{"system": V2_0203, "code": "FILL"}]},
            "value": report_row["filler_report_number"],
        }
    ],
    "custodian": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}, "type": "Organization"},
    "content": [{"attachment": {"contentType": report_row["pdf_content_type"], "url": binary_fullurl}}],
    "context": {
        "period": {"start": report_row["report_datetime"], "end": report_row["report_datetime"]},
        "encounter": [{
            "reference": encounter_fullurl,
            "type": "Encounter",
            "identifier": {"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]},
        }],
        "sourcePatientInfo": {
            "identifier": {
                "assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["mrn_assigner_ods"]}},
                "type": {"coding": [{"system": V2_0203, "code": "MR"}]},
                "value": report_row["mrn"],
            }
        },
        "related": [
            {
                "reference": service_request_fullurl,
                "type": "ServiceRequest",
                "identifier": {
                    "assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}},
                    "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]},
                    "value": report_row["placer_order_number"],
                },
            }
        ],
    },
}

print(json.dumps(document_reference, indent=2))

{
  "resourceType": "DocumentReference",
  "status": "current",
  "type": {
    "coding": [
      {
        "system": "http://snomed.info/sct",
        "code": "1054161000000101",
        "display": "Genetic report"
      }
    ]
  },
  "subject": {
    "reference": "urn:uuid:6e347049-b169-4f19-b3b6-69908b0169dc",
    "identifier": {
      "system": "https://fhir.nhs.uk/Id/nhs-number",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "NH"
          }
        ]
      },
      "value": "9737383222"
    }
  },
  "date": "2026-07-14T15:59:16+00:00",
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699X0"
        }
      },
      "system": "https://fhir.nwgenomics.nhs.uk/iGene/ReportIdentifier",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeS

We don't validate `Binary` on its own — there's nothing profile-specific to check
beyond "is this valid base64", which the JSON parse already guarantees. `DocumentReference`
is worth checking, even though the dangling `urn:uuid` reference to `Binary` will show up
as an unresolved-reference note (expected, same single-resource-validation limitation
`03-order-message-from-csv.ipynb` ran into).

In [14]:
validate_resource(document_reference, DOCUMENT_REFERENCE_PROFILE)

,severity,code,details,expression
0,information,code-invalid,Could not confirm that the codes provided are ...,[DocumentReference.type]


## 9. DiagnosticReport — the resource everything else hangs off

This is the bundle's message *focus*. It's the one resource that references almost
everything else already built: `basedOn` → `ServiceRequest`, `result` → `Observation`,
`presentedForm` → `Binary`. `code` carries both codings the v2 message actually
supplied — the lab's own `IGEAP` code (`OBR-4`) and the Genomic Test Directory code
(`NTE-3`) — since, unlike `ServiceRequest.code`, this repo's real output keeps both here.

In [15]:
diagnostic_report_id = str(uuid4())
diagnostic_report_fullurl = f"urn:uuid:{diagnostic_report_id}"

diagnostic_report = {
    "resourceType": "DiagnosticReport",
    "status": {"F": "final"}.get(report_row["result_status"], "unknown"),
    "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]}],
    "code": {
        "coding": [
            {"system": IGEAP_SYSTEM, "code": report_row["test_code_local"], "display": report_row["test_description"]},
            {"system": GENOMIC_TEST_DIRECTORY_SYSTEM, "code": report_row["test_directory_code"], "display": report_row["test_description"]},
        ]
    },
    "subject": {
        "reference": patient_fullurl,
        "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]},
    },
    "encounter": {
        "reference": encounter_fullurl,
        "identifier": {"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]},
    },
    "effectiveDateTime": report_row["report_datetime"],
    "issued": report_row["report_datetime"],
    "identifier": [
        {
            "assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}},
            "system": IGENE_REPORT_ID_SYSTEM,
            "type": {"coding": [{"system": V2_0203, "code": "FILL"}]},
            "value": report_row["filler_report_number"],
        }
    ],
    "basedOn": [{
        "reference": service_request_fullurl,
        "type": "ServiceRequest",
        "identifier": {
            "assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}},
            "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]},
            "value": report_row["placer_order_number"],
        },
    }],
    "result": [{"reference": observation_fullurl, "type": "Observation", "display": "Variables that apply to the overall study"}],
    "resultsInterpreter": [{"display": f"{report_row['interpreter_given']} {report_row['interpreter_family']}"}],
    "performer": [{"display": GLH_NAME, "identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}, "type": "Organization"}],
    "conclusionCode": [{"coding": [{"system": GENOMIC_TEST_OUTCOME_SYSTEM, "code": report_row["outcome_code"], "display": report_row["outcome_display"]}]}],
    "presentedForm": [{"contentType": report_row["pdf_content_type"], "url": binary_fullurl}],
}

print(json.dumps(diagnostic_report, indent=2))

{
  "resourceType": "DiagnosticReport",
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/v2-0074",
          "code": "GE"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nwgenomics.nhs.uk/CodeSystem/IGEAP",
        "code": "ctDNA_M4",
        "display": "PACKAGE: M4.14 - Non-Small Cell Lung Cancer, Multi-target ctDNA combined Multi-target NGS panel - small variant (EGFR, ALK, BRAF, KRAS, MET exon 14 skipping and copy number variations) and structural variant (ROS1, RET, ALK, NTRK1, NTRK2, NTRK3, MET exon"
      },
      {
        "system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
        "code": "M4.14",
        "display": "PACKAGE: M4.14 - Non-Small Cell Lung Cancer, Multi-target ctDNA combined Multi-target NGS panel - small variant (EGFR, ALK, BRAF, KRAS, MET exon 14 skipping and copy number variations) and structural variant (ROS1, R

In [16]:
validate_resource(diagnostic_report, DIAGNOSTIC_REPORT_PROFILE)

,severity,code,details,expression
5,error,processing,Slicing cannot be evaluated: Could not match d...,[DiagnosticReport.conclusionCode[0].coding[0]]
0,information,informational,This element does not match any known slice de...,[DiagnosticReport.resultsInterpreter[0]]
3,information,informational,This element does not match any known slice de...,[DiagnosticReport.code.coding[0]]
4,information,business-rule,Reference to draft CodeSystem https://fhir.nwg...,[DiagnosticReport.conclusionCode[0]]
6,information,not-supported,DiagnosticReport.conclusionCode.coding:Genomic...,[DiagnosticReport.conclusionCode[0]]
1,warning,code-invalid,Unknown Code 'ctDNA_M4' in the CodeSystem 'htt...,[DiagnosticReport.code.coding[0].code]
2,warning,code-invalid,Unknown Code 'M4.14' in the CodeSystem 'https:...,[DiagnosticReport.code.coding[1].code]


The other `error` row here (`Slicing cannot be evaluated` on
`DiagnosticReport.conclusionCode.coding`) is the same story: it reproduces identically
against this repo's real `/transformToFHIR` output for this message too. It's the
validator reporting that the `DiagnosticReport` profile's own
`conclusionCode.coding:GenomicTestOutcomeCode` slice discriminator (`system`) has no
fixed value, binding, or existence assertion to actually discriminate on — an
under-specified slice definition in the IG itself, not something either conversion gets
wrong. Any `DiagnosticReport` in this IG with a `conclusionCode` hits this; most existing
fixtures in this repo simply don't set one.

## 10. MessageHeader and the whole Bundle

`eventCoding` is `R01`. The report travels the opposite direction to the order in
`03-order-message-from-csv.ipynb`: `sender` is the GLH, `destination` the ordering
trust. As with that notebook's order `Bundle`, `Bundle.identifier` and `Bundle.timestamp`
are mandatory on the
[`Bundle` (message) profile](https://nw-gmsa.github.io/en/StructureDefinition-BundleMessage.html)
and easy to forget since they sit outside every individual resource.

In [17]:
message_header = {
    "resourceType": "MessageHeader",
    "eventCoding": {"system": "http://terminology.hl7.org/CodeSystem/v2-0003", "code": "R01"},
    "sender": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}},
    "destination": [{
        "endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/EPR",
        "receiver": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}},
    }],
    "source": {"endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/HIVE", "software": "NW GLH"},
    "focus": [{"reference": diagnostic_report_fullurl}],
}

bundle_identifier = f"urn:uuid:{uuid4()}"

report_bundle = {
    "resourceType": "Bundle",
    "identifier": {"value": bundle_identifier},
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "type": "message",
    "entry": [
        {"fullUrl": f"urn:uuid:{uuid4()}", "resource": message_header},
        {"fullUrl": patient_fullurl, "resource": patient},
        {"fullUrl": encounter_fullurl, "resource": encounter},
        {"fullUrl": service_request_fullurl, "resource": service_request},
        {"fullUrl": observation_fullurl, "resource": observation},
        {"fullUrl": binary_fullurl, "resource": binary},
        {"fullUrl": document_reference_fullurl, "resource": document_reference},
        {"fullUrl": diagnostic_report_fullurl, "resource": diagnostic_report},
    ],
}

report_filename = "ctdna9737383222-handbuilt.json"
with open("Input/FHIR/R01/" + report_filename, "w") as f:
    json.dump(report_bundle, f, indent=2)

print("Saved Input/FHIR/R01/" + report_filename)

Saved Input/FHIR/R01/ctdna9737383222-handbuilt.json


In [18]:
subprocess.run(
    [
        "java", "-jar", "validator_cli.jar", "Input/FHIR/R01/" + report_filename,
        "-version", "4.0.1", "-ig", "package.tgz",
        "-bundle", "DiagnosticReport:0", DIAGNOSTIC_REPORT_PROFILE, "-tx", "n/a",
        "-output", "Results/FHIR/R01/" + report_filename + "-OperationOutcome.json",
        "-output-style", "json",
    ],
    capture_output=True,
)

with open("Results/FHIR/R01/" + report_filename + "-OperationOutcome.json") as f:
    outcome = json.load(f)

issues_df(outcome)

,severity,code,details,expression
34,error,processing,Slicing cannot be evaluated: Could not match d...,[Bundle.entry[7].resource.conclusionCode[0].co...
18,information,informational,This element does not match any known slice de...,[Bundle.entry[4].resource/*Observation/null*/....
19,information,code-invalid,The value provided ('application/pdf') could n...,[Bundle.entry[5].resource/*Binary/null*/.conte...
17,information,informational,This element does not match any known slice de...,[Bundle.entry[4].resource/*Observation/null*/....
16,information,informational,This element does not match any known slice de...,[Bundle.entry[4].resource/*Observation/null*/....
15,information,informational,This element does not match any known slice de...,[Bundle.entry[4].resource/*Observation/null*/....
13,information,business-rule,Reference to draft CodeSystem https://fhir.nwg...,[Bundle.entry[4].resource/*Observation/null*/....
27,information,business-rule,Reference to draft CodeSystem https://fhir.nwg...,[Bundle.entry[7].resource/*DiagnosticReport/nu...
28,information,code-invalid,The value provided ('application/pdf') could n...,[Bundle.entry[7].resource/*DiagnosticReport/nu...
11,information,business-rule,Reference to draft CodeSystem https://fhir.nwg...,[Bundle.entry[4].resource/*Observation/null*/....


## 11. Convert it to HL7 v2 — `transformToV2`

Feeding our hand-built `Bundle` through the same tooling endpoint
`03-order-message-from-csv.ipynb` used shows this repo's real transform engine can round
-trip data it never produced itself. Because NW Genomics' v2 (2.5.1) and FHIR profiles
share the same canonical model, this direction (FHIR → v2) needs no extra logic beyond
"render the resources as segments" — there's no re-derivation of codes/identifiers the
way there was going v2 → FHIR in sections 4-9 above.

In [19]:
headersFHIR = {"Content-Type": "application/fhir+json"}

with open("Input/FHIR/R01/" + report_filename, "rb") as f:
    report_json = f.read()

rV2 = requests.post(toolsServer + "/transformToV2", data=report_json, verify=False, headers=headersFHIR)

with open("Output/V2/R01/" + report_filename.replace(".json", ".txt"), "w") as f:
    f.write(rV2.text)

print(rV2.text)

## 12. Compare against `/transformToFHIR`

Now, and only now, we call the API this notebook otherwise avoided — to check our
hand-built resources against what this repo's real transformation engine produces from
the *same original* `ctdna9737383222.txt`. The two won't be byte-identical (fresh
`urn:uuid`s every call, and our version skips a couple of niceties like the
verification-status extension on the NHS number), but the resource types, codes, and
identifiers should line up.

In [20]:
with open("Input/V2/R01/ctdna9737383222.txt", "rb") as f:
    v2_bytes = f.read()

headersV2 = {"Content-Type": "x-application/hl7-v2+er7"}
rFHIR = requests.post(toolsServer + "/transformToFHIR", data=v2_bytes, verify=False, headers=headersV2)
reference_bundle = rFHIR.json()

ours = sorted(e["resource"]["resourceType"] for e in report_bundle["entry"])
theirs = sorted(e["resource"]["resourceType"] for e in reference_bundle["entry"])
print("Our resource types:      ", ours)
print("transformToFHIR's types: ", theirs)

Our resource types:       ['Binary', 'DiagnosticReport', 'DocumentReference', 'Encounter', 'MessageHeader', 'Observation', 'Patient', 'ServiceRequest']
transformToFHIR's types:  ['Binary', 'DiagnosticReport', 'DocumentReference', 'Encounter', 'MessageHeader', 'Observation', 'Patient', 'ServiceRequest']


## 13. From R01 to T02 — the same resources, a different message

A ctDNA report isn't only an `ORU^R01` result to the ordering trust — the shared-care
-record providers described below want to be *notified a document exists*, which HL7 v2
models as `MDM^T02` rather than `ORU^R01`. This repo's own established practice for
producing a `T02` fixture from an `R01` one (`Testing.ipynb`) is not to rebuild the
bundle - the `DocumentReference`/`Binary` pair is already sitting right there in the
`R01` bundle - it just flips `MessageHeader.eventCoding.code` from `R01` to `T02`. NW
Genomics' v2 and FHIR profiles sharing one canonical model is exactly what makes that
minimal a change sufficient; we do the same here rather than inventing a different
approach.

In [21]:
import copy

t02_bundle = copy.deepcopy(report_bundle)
for entry in t02_bundle["entry"]:
    if entry["resource"]["resourceType"] == "MessageHeader":
        entry["resource"]["eventCoding"]["code"] = "T02"

t02_filename = "MDM_T02_ctdna9737383222-handbuilt.json"
with open("Output/FHIR/T02/" + t02_filename, "w") as f:
    json.dump(t02_bundle, f, indent=2)

print("Saved Output/FHIR/T02/" + t02_filename)

Saved Output/FHIR/T02/MDM_T02_ctdna9737383222-handbuilt.json


## 14. MDM^T02 for NHS Trusts and other HL7 v2 shared-care-record providers

Some shared-care-record providers, like NHS Trusts, are happiest with HL7 v2 — so the
`T02` bundle goes through the exact same `transformToV2` endpoint as section 11.

In [22]:
with open("Output/FHIR/T02/" + t02_filename, "rb") as f:
    t02_json = f.read()

rV2T02 = requests.post(toolsServer + "/transformToV2", data=t02_json, verify=False, headers=headersFHIR)

with open("Output/V2/T02/" + t02_filename.replace(".json", ".txt"), "w") as f:
    f.write(rV2T02.text)

print(rV2T02.text)

This repo already has a real `MDM^T02` fixture for this same patient
(`Output/V2/T02/MDM_T02_ctdna9737383222.txt`, produced via `/transformToFHIR` +
`/transformToV2` rather than by hand) — worth a quick look alongside ours as a sanity
check, even though (as in section 12) they won't match byte-for-byte.

In [23]:
with open("Output/V2/T02/MDM_T02_ctdna9737383222.txt") as f:
    print(f.read())

MSH|^~\&|IGENE|699X0|EPR|699X0|20260709130251+0000||MDM^T02|IGENE:MFT:c61c188b-4843-4b1f-8c84-12af80be0568|T|2.4|||AL
EVN|T02|20260709130251+0000
PID|1||9737383222^^^NHS^NH||LEEDS^Rob^^^^^L||19780117|M|||^^^^LS1 3EX|||||||||||||||||||||05
PV1|1|O|||||||||||||||||SP26-01847^^^^AN
TXA|1|1054161000000101^Genetic report^SNM3||||20260714155916+0000||||||T26-59X2
OBX|1|ED|1054161000000101^Genetic report^SNM3||^application^pdf^Base64^JVBERi0xLjQKMSAwIG9iago8PC9UeXBlIC9DYXRhbG9nCi9QYWdlcyAyIDAgUgo+PgplbmRvYmoK MiAwIG9iago8PC9UeXBlIC9QYWdlcwovS2lkcyBbMyAwIFJdCi9Db3VudCAxCj4+CmVuZG9iagozIDAgb2JqCjw8L1R5cGUgL1BhZ2UKL1BhcmVudCAyIDAgUgovTWVkaWFCb3ggWzAgMCA1OTUgODQy XQovQ29udGVudHMgNSAwIFIKL1Jlc291cmNlcyA8PC9Qcm9jU2V0IFsvUERGIC9UZXh0XQovRm9udCA8PC9GMSA0IDAgUj4+Cj4+Cj4+CmVuZG9iago0IDAgb2JqCjw8L1R5cGUgL0ZvbnQKL1N1YnR5 cGUgL1R5cGUxCi9OYW1lIC9GMQovQmFzZUZvbnQgL0hlbHZldGljYQovRW5jb2RpbmcgL01hY1JvbWFuRW5jb2RpbmcKPj4KZW5kb2JqCjUgMCBvYmoKPDwvTGVuZ3RoIDUzCj4+CnN0cmVhbQpCVAov RjEgMjAgVGYKMjIwIDQwMCBUZAooRHVtb

## 15. FHIR shared-care-record providers — IHE MHD ITI-105

Other shared-care-record providers want FHIR rather than v2 — [IHE MHD's
`ITI-105` "Provide Document Bundle" (Simplified Publish)](https://profiles.ihe.net/ITI/MHD/ITI-105.html)
is a common shape for that. Unlike NW-GMSA's own `Bundle` (message), `ITI-105` isn't a
bundle transaction at all: it's a plain FHIR `create` — an HTTP `POST` of a single
`DocumentReference` straight to `[base]/DocumentReference`, with the document's bytes
inlined into `DocumentReference.content.attachment.data` rather than referenced by `url`
the way our own `T02` bundle's `DocumentReference` points at its sibling `Binary`.

Building the `ITI-105` shape from our `T02` bundle is a small, mechanical conversion:

- inline the `Binary`'s `data` into `content.attachment.data`, instead of an `url`
  reference to it
- add `content.attachment.hash` (base64-encoded SHA-1 of the raw bytes) and `.size`,
  both mandatory alongside inline `data` on this profile
- `ITI-105` expects `subject` to point at a "commonly accessible" `Patient` — since
  there's no shared FHIR server both sides can resolve a logical reference against
  here, we `contain` the `Patient` instead and reference it locally (`#patient`), the
  profile's documented alternative to an external reference

There's no real MHD endpoint to `POST` this to in this exercise, so we stop at building
the resource — the last line below is the request a real Document Source would send.

In [24]:
t02_document_reference = next(
    e["resource"] for e in t02_bundle["entry"] if e["resource"]["resourceType"] == "DocumentReference"
)
t02_binary = next(e["resource"] for e in t02_bundle["entry"] if e["resource"]["resourceType"] == "Binary")

pdf_bytes = base64.b64decode(t02_binary["data"])
pdf_hash = base64.b64encode(hashlib.sha1(pdf_bytes).digest()).decode("ascii")

iti105_document_reference = copy.deepcopy(t02_document_reference)
iti105_document_reference["contained"] = [{**patient, "id": "patient"}]
iti105_document_reference["subject"] = {"reference": "#patient"}
iti105_document_reference["content"][0]["attachment"] = {
    "contentType": t02_binary["contentType"],
    "data": t02_binary["data"],
    "hash": pdf_hash,
    "size": len(pdf_bytes),
}

print(json.dumps(iti105_document_reference, indent=2)[:2000])

{
  "resourceType": "DocumentReference",
  "status": "current",
  "type": {
    "coding": [
      {
        "system": "http://snomed.info/sct",
        "code": "1054161000000101",
        "display": "Genetic report"
      }
    ]
  },
  "subject": {
    "reference": "#patient"
  },
  "date": "2026-07-14T15:59:16+00:00",
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699X0"
        }
      },
      "system": "https://fhir.nwgenomics.nhs.uk/iGene/ReportIdentifier",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "FILL"
          }
        ]
      },
      "value": "T26-59X2"
    }
  ],
  "custodian": {
    "identifier": {
      "system": "https://fhir.nhs.uk/Id/ods-organization-code",
      "value": "699X0"
    },
    "type": "Organization"
  },
  "content": [
    {
      "attachment": {
   

```
POST [base]/DocumentReference
Content-Type: application/fhir+json
Accept: application/fhir+json

<iti105_document_reference>
```

## Summary

Starting from a raw HL7 v2 `ORU^R01`, we hand-parsed its segments into a plain dict (the
same shape a SQL query would produce), built and validated a `Patient`/`Encounter`/
`ServiceRequest`/`Observation`/`Binary`/`DocumentReference`/`DiagnosticReport` `Bundle`
from it, and checked our own conversion against this repo's real `/transformToFHIR`. We
then converted that `Bundle` to HL7 v2 via `transformToV2`, turned it into an `MDM^T02`
document notification the same way `Testing.ipynb` does (flip `eventCoding`, no rebuild
needed), converted *that* to v2 for HL7-speaking shared-care-record providers, and built
an IHE MHD `ITI-105`-shaped `DocumentReference` for FHIR-speaking ones.